# SC-G2-A — Kiểm toán Dữ liệu Nguồn Scania APS (Source Audit Notebook)

**Dự án:** Tối ưu chi phí bảo trì dự đoán bằng Machine Learning  
**Dataset:** APS Failure at Scania Trucks (UCI ID: 421, DOI: 10.24432/C51S51)  
**Mã nhiệm vụ:** SC-G2-A (Antigravity Data Audit)  
**Mục tiêu:** Kiểm toán tính toàn vẹn (integrity), cấu trúc file, schema, nhãn, phân bố khuyết thiếu (missingness), thuộc tính hằng số và trùng lặp (exact-X duplicates) trên tập Train chính thức 60.000 mẫu (`aps_failure_training_set.csv`).  
**Ranh giới & Quy tắc bảo vệ dữ liệu:**
- Tập Test 16.000 mẫu (`aps_failure_test_set.csv`) được niêm phong (sealed) bên trong archive gốc, tuyệt đối không trích xuất, mở hoặc khảo sát trước G10.
- Không chia Train/Validation trong G2, không fit imputer/scaler, không lọc đặc trưng theo nhãn, không huấn luyện mô hình.
- Không âm thầm sửa dữ liệu lỗi. Notebook có thể tái lập 100% độc lập từ raw Train.

In [1]:
import os
import sys
import hashlib
import math
import csv
import json
import platform
import time
import datetime
from collections import Counter, defaultdict

# Xác định đường dẫn gốc của repository một cách linh hoạt (không phụ thuộc máy cá nhân)
cwd = os.path.abspath(os.getcwd())
if os.path.exists(os.path.join(cwd, "data", "raw", "scania_aps")):
    REPO_ROOT = cwd
elif os.path.exists(os.path.join(cwd, "..", "..", "data", "raw", "scania_aps")):
    REPO_ROOT = os.path.abspath(os.path.join(cwd, "..", ".."))
else:
    raise FileNotFoundError("Không thể xác định thư mục gốc của repository từ: " + cwd)

RAW_DIR = os.path.join(REPO_ROOT, "data", "raw", "scania_aps")
TRAIN_CSV = os.path.join(RAW_DIR, "aps_failure_training_set.csv")
RESULTS_DIR = os.path.join(REPO_ROOT, "results", "scania_aps", "g2_audit")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Thư mục Repository Root: {REPO_ROOT}")
print(f"Tệp Train cần kiểm toán: {TRAIN_CSV}")
print(f"Thư mục xuất kết quả G2:  {RESULTS_DIR}")
assert os.path.exists(TRAIN_CSV), f"LỖI: Không tìm thấy tệp {TRAIN_CSV}"

Thư mục Repository Root: D:\Năm 3\HK1_2026\Nhập môn tối ưu hóa\baitaplon\worktrees\antigravity
Tệp Train cần kiểm toán: D:\Năm 3\HK1_2026\Nhập môn tối ưu hóa\baitaplon\worktrees\antigravity\data\raw\scania_aps\aps_failure_training_set.csv
Thư mục xuất kết quả G2:  D:\Năm 3\HK1_2026\Nhập môn tối ưu hóa\baitaplon\worktrees\antigravity\results\scania_aps\g2_audit


## 1. Kiểm toán Provenance & Tính Toàn vẹn Nhị phân (Binary Integrity & SHA-256)

Xác thực mã băm SHA-256 và kích thước byte chính xác của tệp Train thô.

In [2]:
sha256_calc = hashlib.sha256()
file_size = os.path.getsize(TRAIN_CSV)

with open(TRAIN_CSV, "rb") as f:
    while chunk := f.read(65536):
        sha256_calc.update(chunk)
train_sha256 = sha256_calc.hexdigest()

EXPECTED_TRAIN_SIZE = 44669194
EXPECTED_TRAIN_SHA256 = "bb484302e3a3a1c8ef5e1f0129c4dc7cbd58f350867f95b575461ca21ab6b9da"

print(f"Dung lượng tệp Train: {file_size:,} bytes (kỳ vọng: {EXPECTED_TRAIN_SIZE:,})")
print(f"Mã SHA-256 Train:   {train_sha256}")
assert file_size == EXPECTED_TRAIN_SIZE, f"Sai lệch kích thước tệp: {file_size} != {EXPECTED_TRAIN_SIZE}"
assert train_sha256 == EXPECTED_TRAIN_SHA256, f"Sai lệch SHA-256: {train_sha256} != {EXPECTED_TRAIN_SHA256}"
print("=> KIỂM TRA TOÀN VẸN BYTE & HASH: PASS 100%")

Dung lượng tệp Train: 44,669,194 bytes (kỳ vọng: 44,669,194)
Mã SHA-256 Train:   bb484302e3a3a1c8ef5e1f0129c4dc7cbd58f350867f95b575461ca21ab6b9da
=> KIỂM TRA TOÀN VẸN BYTE & HASH: PASS 100%


## 2. Kiểm toán Strict Encoding & Phát hiện Header Không Hard-code

- Kiểm tra BOM nhị phân (BOM check) để đảm bảo không bị ô nhiễm byte ẩn.
- Kiểm tra decode UTF-8 nghiêm ngặt trên toàn bộ tệp (strict UTF-8 decode, không dùng `errors='replace'`).
- Tự động quét cấu trúc để phát hiện dòng Header chứa `class` mà không hard-code số dòng bỏ qua.

In [3]:
# Kiểm tra BOM nhị phân
with open(TRAIN_CSV, "rb") as f:
    first_bytes = f.read(4)
if first_bytes.startswith(b"\xef\xbb\xbf"):
    bom_status = "UTF-8 with BOM"
elif first_bytes.startswith(b"\xff\xfe") or first_bytes.startswith(b"\xfe\xff"):
    bom_status = "UTF-16"
else:
    bom_status = "Standard UTF-8 / ASCII (No BOM)"

print(f"Trạng thái Byte-Order Mark: {bom_status}")
assert bom_status == "Standard UTF-8 / ASCII (No BOM)", f"LỖI: Phát hiện BOM lạ {bom_status}"

# Quét preamble và phát hiện header theo cấu trúc dòng thực tế
preamble_lines = []
header_line_index = None
header = None

with open(TRAIN_CSV, "r", encoding="utf-8", errors="strict") as f:
    for idx, line in enumerate(f):
        stripped = line.strip()
        parts = [p.strip() for p in stripped.split(",")]
        if len(parts) > 1 and parts[0] == "class":
            header_line_index = idx
            header = parts
            break
        else:
            preamble_lines.append(stripped)

assert header is not None, "LỖI: Không tìm thấy dòng header chứa cột 'class'!"
print(f"Số dòng chú thích đầu tệp (preamble): {len(preamble_lines)} dòng")
print(f"Vị trí dòng header: Dòng {header_line_index + 1} (chỉ số 0-based: {header_line_index})")
print(f"Tổng số cột trong header: {len(header)}")
assert len(header) == 171, f"LỖI: Kỳ vọng 171 cột nhưng nhận được {len(header)}"
assert header[0] == "class", f"LỖI: Cột đầu tiên phải là 'class', thực tế: '{header[0]}'"
assert len(set(header)) == 171, "LỖI: Tồn tại tên cột bị trùng lặp trong header!"

feature_names = header[1:]
print(f"Số lượng cột đặc trưng vận hành: {len(feature_names)}")
print(f"5 đặc trưng đầu: {feature_names[:5]}")
print(f"5 đặc trưng cuối: {feature_names[-5:]}")
print("=> KIỂM TOÁN ENCODING & HEADER: PASS 100%")

Trạng thái Byte-Order Mark: Standard UTF-8 / ASCII (No BOM)
Số dòng chú thích đầu tệp (preamble): 20 dòng
Vị trí dòng header: Dòng 21 (chỉ số 0-based: 20)
Tổng số cột trong header: 171
Số lượng cột đặc trưng vận hành: 170
5 đặc trưng đầu: ['aa_000', 'ab_000', 'ac_000', 'ad_000', 'ae_000']
5 đặc trưng cuối: ['ee_007', 'ee_008', 'ee_009', 'ef_000', 'eg_000']
=> KIỂM TOÁN ENCODING & HEADER: PASS 100%


## 3. Kiểm toán Dữ liệu Dòng, Biến Mục tiêu & Token Khuyết thiếu Strict

- Kiểm tra từng dòng: đủ đúng 171 cột (không có dòng thiếu/thừa trường).
- Kiểm tra cột nhãn `class`: chỉ chấp nhận giá trị nghiêm ngặt là `pos` hoặc `neg`. Bất kỳ token lạ nào khác đều kích hoạt lỗi dừng.
- Kiểm tra token khuyết thiếu: chỉ chấp nhận duy nhất chuỗi `na`.
- Kiểm tra giá trị số: tất cả các ô khác `na` phải parse được thành số thực hữu hạn (finite float). Nghiêm cấm các token bất thường như `nan`, `inf`, `-inf`, `null`, v.v.
- Thiết lập định danh nguồn không trùng lặp `source_row_id` theo đúng hợp đồng P01.

In [4]:
FORBIDDEN_TOKENS = {"nan", "+nan", "-nan", "inf", "+inf", "-inf", "null", "none", "?", ""}

row_count = 0
malformed_rows = 0
target_counts = Counter()
missing_counts = [0] * len(feature_names)
observed_counts = [0] * len(feature_names)
col_min = [float("inf")] * len(feature_names)
col_max = [float("-inf")] * len(feature_names)
col_sum = [0.0] * len(feature_names)

row_missing_distribution = Counter()
rows_all_missing = 0

# Cấu trúc dữ liệu phục vụ kiểm toán trùng lặp P03
exact_x_map = defaultdict(list)
source_row_ids = []

t0 = time.time()

with open(TRAIN_CSV, "r", encoding="utf-8", errors="strict") as f:
    # Bỏ qua preamble và header
    for _ in range(header_line_index + 1):
        next(f)
    
    csv_reader = csv.reader(f)
    for row in csv_reader:
        row_count += 1
        data_row_number = row_count
        
        if len(row) != 171:
            malformed_rows += 1
            raise ValueError(f"Dòng {data_row_number} có {len(row)} cột, kỳ vọng 171!")
        
        # Kiểm toán biến mục tiêu
        raw_target = row[0].strip()
        if raw_target == "pos":
            mapped_target = 1
            target_counts["pos"] += 1
        elif raw_target == "neg":
            mapped_target = 0
            target_counts["neg"] += 1
        else:
            raise ValueError(f"Dòng {data_row_number}: Nhãn mục tiêu không hợp lệ '{raw_target}'!")
        
        # Hợp đồng định danh dòng nguồn source_row_id (P01)
        s_row_id = (train_sha256, "train", data_row_number)
        source_row_ids.append(s_row_id)
        
        row_missing = 0
        canonical_x = []
        
        # Kiểm toán 170 thuộc tính vận hành
        for col_idx, raw_cell in enumerate(row[1:]):
            val_str = raw_cell.strip()
            
            if val_str == "na":
                missing_counts[col_idx] += 1
                row_missing += 1
                canonical_x.append(None)
            else:
                val_lower = val_str.lower()
                if val_lower in FORBIDDEN_TOKENS:
                    raise ValueError(f"Dòng {data_row_number}, cột {feature_names[col_idx]}: Phát hiện token cấm '{val_str}'!")
                
                try:
                    val_float = float(val_str)
                except ValueError:
                    raise ValueError(f"Dòng {data_row_number}, cột {feature_names[col_idx]}: Không thể chuyển đổi sang float '{val_str}'!")
                
                if not math.isfinite(val_float):
                    raise ValueError(f"Dòng {data_row_number}, cột {feature_names[col_idx]}: Giá trị non-finite {val_float}!")
                
                observed_counts[col_idx] += 1
                col_sum[col_idx] += val_float
                if val_float < col_min[col_idx]:
                    col_min[col_idx] = val_float
                if val_float > col_max[col_idx]:
                    col_max[col_idx] = val_float
                
                canonical_x.append(val_float)
        
        row_missing_distribution[row_missing] += 1
        if row_missing == len(feature_names):
            rows_all_missing += 1
        
        # Lưu trữ vector X cho kiểm toán trùng lặp P03
        exact_x_map[tuple(canonical_x)].append((data_row_number, mapped_target))

elapsed = time.time() - t0
print(f"Hoàn thành duyệt toàn bộ dữ liệu trong {elapsed:.2f} giây.")
print(f"Tổng số dòng dữ liệu: {row_count:,}")
print(f"Số dòng dị dạng:     {malformed_rows}")
print(f"Phân bố nhãn:        {dict(target_counts)}")
assert row_count == 60000, f"LỖI: Kỳ vọng 60.000 dòng, thực tế {row_count}"
assert target_counts["neg"] == 59000, f"LỖI: Kỳ vọng 59.000 neg, thực tế {target_counts['neg']}"
assert target_counts["pos"] == 1000, f"LỖI: Kỳ vọng 1.000 pos, thực tế {target_counts['pos']}"
assert malformed_rows == 0, "LỖI: Tồn tại dòng dị dạng!"
print("=> KIỂM TOÁN NHÃN VÀ DỮ LIỆU DÒNG: PASS 100%")

Hoàn thành duyệt toàn bộ dữ liệu trong 9.56 giây.
Tổng số dòng dữ liệu: 60,000
Số dòng dị dạng:     0
Phân bố nhãn:        {'neg': 59000, 'pos': 1000}
=> KIỂM TOÁN NHÃN VÀ DỮ LIỆU DÒNG: PASS 100%


## 4. Kiểm toán Trùng lặp Exact-X & Xung đột Nhãn (Phê duyệt P03)

- Kiểm toán exact-X duplicate trước khi điền khuyết (imputation) bằng 170 giá trị numeric đã parse và missing mask.
- Loại trừ hoàn toàn cột nhãn `class` và `source_row_id`.
- Thống kê số nhóm trùng lặp, số dòng liên quan và các trường hợp xung đột nhãn (cùng $X$ nhưng khác nhãn).
- Xác lập trạng thái cổng P03: `PASSED` nếu không có trùng lặp; `BLOCKED` nếu phát hiện duplicate/conflict.

In [5]:
unique_patterns = len(exact_x_map)
duplicate_groups = sum(1 for rows in exact_x_map.values() if len(rows) > 1)
total_duplicate_rows = sum(len(rows) for rows in exact_x_map.values() if len(rows) > 1)
surplus_duplicate_rows = sum(len(rows) - 1 for rows in exact_x_map.values() if len(rows) > 1)

conflicting_groups = []
for x_pat, rows in exact_x_map.items():
    if len(rows) > 1:
        distinct_labels = set(lbl for _, lbl in rows)
        if len(distinct_labels) > 1:
            conflicting_groups.append({
                "size": len(rows),
                "rows_pos": [rn for rn, l in rows if l == 1],
                "rows_neg": [rn for rn, l in rows if l == 0]
            })

print(f"Tổng số mẫu đặc trưng X độc nhất: {unique_patterns:,} / {row_count:,}")
print(f"Số nhóm trùng lặp (duplicate groups): {duplicate_groups}")
print(f"Tổng số dòng trong nhóm trùng lặp:   {total_duplicate_rows}")
print(f"Số dòng dư thừa (surplus duplicates): {surplus_duplicate_rows}")
print(f"Số nhóm xung đột nhãn (label conflicts): {len(conflicting_groups)}")

assert unique_patterns == 60000, f"LỖI: Kỳ vọng 60.000 unique patterns, nhận được {unique_patterns}"
assert duplicate_groups == 0, f"LỖI: Phát hiện {duplicate_groups} nhóm trùng lặp!"
assert len(conflicting_groups) == 0, f"LỖI: Phát hiện {len(conflicting_groups)} nhóm xung đột nhãn!"

p03_status = "PASSED" if (duplicate_groups == 0 and len(conflicting_groups) == 0) else "BLOCKED"
print(f"=> TRẠNG THÁI CỔNG P03: {p03_status}")

Tổng số mẫu đặc trưng X độc nhất: 60,000 / 60,000
Số nhóm trùng lặp (duplicate groups): 0
Tổng số dòng trong nhóm trùng lặp:   0
Số dòng dư thừa (surplus duplicates): 0
Số nhóm xung đột nhãn (label conflicts): 0
=> TRẠNG THÁI CỔNG P03: PASSED


## 5. Phân tích Chi tiết Khuyết thiếu (Missingness Profiling) & Thuộc tính Hằng số

- Tính toán tổng số lượng ô missing trên toàn tập ($170 \times 60.000$).
- Thống kê tỷ lệ khuyết thiếu theo từng cột: nhận diện các cột không thiếu (0%), thiếu cực đoan (>50%, >70%, >80%), và thiếu hoàn toàn (100%).
- Thống kê khuyết thiếu theo từng dòng: min missing, max missing, dòng all-missing (170/170).
- Kiểm toán thuộc tính hằng số / phương sai bằng 0 (Zero-Variance) giữa các giá trị quan sát được.

In [6]:
total_cells = len(feature_names) * row_count
total_missing = sum(missing_counts)
total_observed = sum(observed_counts)
overall_missing_rate = total_missing / total_cells

print(f"Tổng số ô dữ liệu đặc trưng: {total_cells:,}")
print(f"Tổng số ô khuyết thiếu:     {total_missing:,} ({overall_missing_rate:.4%})")
print(f"Tổng số ô quan sát được:     {total_observed:,} ({total_observed / total_cells:.4%})")
assert total_missing == 850015, f"LỖI: Kỳ vọng 850.015 ô missing, thực tế {total_missing}"
assert total_observed == 9349985, f"LỖI: Kỳ vọng 9.349.985 ô observed, thực tế {total_observed}"

# Missing theo cột
col_missing_rates = [c / row_count for c in missing_counts]
zero_missing_cols = [feature_names[i] for i, r in enumerate(col_missing_rates) if r == 0.0]
all_missing_cols = [feature_names[i] for i, r in enumerate(col_missing_rates) if r == 1.0]
gt_50_cols = [(feature_names[i], missing_counts[i], col_missing_rates[i]) for i, r in enumerate(col_missing_rates) if r > 0.50]
gt_70_cols = [(feature_names[i], missing_counts[i], col_missing_rates[i]) for i, r in enumerate(col_missing_rates) if r > 0.70]
gt_80_cols = [(feature_names[i], missing_counts[i], col_missing_rates[i]) for i, r in enumerate(col_missing_rates) if r > 0.80]

gt_50_cols.sort(key=lambda x: x[2], reverse=True)

print(f"\nSố cột không khuyết thiếu (0% missing): {len(zero_missing_cols)} ({zero_missing_cols})")
print(f"Số cột khuyết thiếu hoàn toàn (100% missing): {len(all_missing_cols)}")
print(f"Số cột có tỷ lệ missing > 50%: {len(gt_50_cols)}")
print(f"Số cột có tỷ lệ missing > 70%: {len(gt_70_cols)}")
print(f"Số cột có tỷ lệ missing > 80%: {len(gt_80_cols)}")

print("\nDanh sách 8 thuộc tính có tỷ lệ missing > 50%:")
for fn, cnt, r in gt_50_cols:
    print(f"  - {fn}: {cnt:,} / {row_count:,} ({r:.2%})")

assert len(zero_missing_cols) == 1 and zero_missing_cols[0] == "aa_000"
assert len(all_missing_cols) == 0
assert len(gt_50_cols) == 8
assert len(gt_70_cols) == 7
assert len(gt_80_cols) == 2

# Missing theo dòng
min_missing_row = min(row_missing_distribution.keys())
max_missing_row = max(row_missing_distribution.keys())
print(f"\nSố missing ít nhất trên một dòng: {min_missing_row}")
print(f"Số missing nhiều nhất trên một dòng: {max_missing_row}")
print(f"Số dòng thiếu toàn bộ 170 thuộc tính: {rows_all_missing}")
assert min_missing_row == 0
assert max_missing_row == 168
assert rows_all_missing == 0

# Kiểm toán thuộc tính hằng số (Zero-Variance)
constant_cols = []
for i, fn in enumerate(feature_names):
    if observed_counts[i] > 0 and col_min[i] == col_max[i]:
        constant_cols.append((fn, col_min[i], observed_counts[i], missing_counts[i]))

print(f"\nSố thuộc tính hằng số (phương sai = 0 giữa các giá trị quan sát): {len(constant_cols)}")
for fn, val, obs, mis in constant_cols:
    print(f"  - {fn}: giá trị hằng số {val} (quan sát {obs:,} dòng, khuyết {mis:,} dòng)")
assert len(constant_cols) == 1
assert constant_cols[0][0] == "cd_000" and constant_cols[0][1] == 1209600.0
print("=> KIỂM TOÁN MISSINGNESS VÀ CONSTANT FEATURES: PASS 100%")

Tổng số ô dữ liệu đặc trưng: 10,200,000
Tổng số ô khuyết thiếu:     850,015 (8.3335%)
Tổng số ô quan sát được:     9,349,985 (91.6665%)

Số cột không khuyết thiếu (0% missing): 1 (['aa_000'])
Số cột khuyết thiếu hoàn toàn (100% missing): 0
Số cột có tỷ lệ missing > 50%: 8
Số cột có tỷ lệ missing > 70%: 7
Số cột có tỷ lệ missing > 80%: 2

Danh sách 8 thuộc tính có tỷ lệ missing > 50%:
  - br_000: 49,264 / 60,000 (82.11%)
  - bq_000: 48,722 / 60,000 (81.20%)
  - bp_000: 47,740 / 60,000 (79.57%)
  - bo_000: 46,333 / 60,000 (77.22%)
  - ab_000: 46,329 / 60,000 (77.22%)
  - cr_000: 46,329 / 60,000 (77.22%)
  - bn_000: 44,009 / 60,000 (73.35%)
  - bm_000: 39,549 / 60,000 (65.92%)

Số missing ít nhất trên một dòng: 0
Số missing nhiều nhất trên một dòng: 168
Số dòng thiếu toàn bộ 170 thuộc tính: 0

Số thuộc tính hằng số (phương sai = 0 giữa các giá trị quan sát): 1
  - cd_000: giá trị hằng số 1209600.0 (quan sát 59,324 dòng, khuyết 676 dòng)
=> KIỂM TOÁN MISSINGNESS VÀ CONSTANT FEATURES: PASS 

## 6. Xác minh 7 Họ Đặc trưng dạng Histogram

Xác thực cấu trúc mô tả từ UCI về 7 họ biến histogram (mỗi họ gồm 10 bins từ 000 đến 009, tổng 70 thuộc tính) và 100 thuộc tính counters.

In [7]:
hist_prefixes = ['ag', 'ay', 'az', 'ba', 'cn', 'cs', 'ee']
histogram_features_count = 0

for pfx in hist_prefixes:
    bins = [fn for fn in feature_names if fn.startswith(f"{pfx}_")]
    assert len(bins) == 10, f"LỖI: Họ {pfx} không đủ 10 bins, nhận được {len(bins)}"
    expected_bins = [f"{pfx}_{i:03d}" for i in range(10)]
    assert bins == expected_bins, f"LỖI: Thứ tự bins của {pfx} không đúng chuẩn"
    histogram_features_count += len(bins)

counter_features_count = len(feature_names) - histogram_features_count
print(f"Tổng số thuộc tính dạng Histogram: {histogram_features_count} (7 họ x 10 bins)")
print(f"Tổng số thuộc tính dạng Counter:   {counter_features_count}")
print(f"Tổng số thuộc tính vận hành:       {len(feature_names)}")
assert histogram_features_count == 70
assert counter_features_count == 100
print("=> XÁC MINH CẤU TRÚC HISTOGRAM: PASS 100%")

Tổng số thuộc tính dạng Histogram: 70 (7 họ x 10 bins)
Tổng số thuộc tính dạng Counter:   100
Tổng số thuộc tính vận hành:       170
=> XÁC MINH CẤU TRÚC HISTOGRAM: PASS 100%


## 7. Tự động Tạo / Tái tạo Toàn bộ Artifacts Kiểm toán G2

Thực thi việc xuất các tệp artifacts theo đúng hợp đồng phân quyền SC-G2-A:
1. `acquisition_manifest.json`: Lưu vết xuất xứ, DOI, URL, hash các tệp thô.
2. `audit_summary.json`: Tóm tắt các chỉ số kiểm toán, phân bố nhãn, missing, zero-variance và cổng P01/P03.
3. `feature_profiles.json`: Profile thống kê chi tiết cho toàn bộ 170 thuộc tính.
4. `environment.json`: Snapshot cấu hình môi trường thực thi hiện tại.
5. `audit_execution.txt`: Nhật ký ghi nhận thời gian chạy và lệnh thực thi.

In [8]:
# 1. Xuất feature_profiles.json
feature_profiles = []
hist_pfx_set = set(hist_prefixes)
for i, fn in enumerate(feature_names):
    pfx = fn.split("_")[0]
    obs = observed_counts[i]
    mis = missing_counts[i]
    mean_val = (col_sum[i] / obs) if obs > 0 else None
    is_const = (obs > 0 and col_min[i] == col_max[i])
    feature_profiles.append({
        "feature_name": fn,
        "prefix": pfx,
        "type": "histogram_bin" if pfx in hist_pfx_set else "numerical_counter",
        "observed_count": obs,
        "missing_count": mis,
        "missing_rate": mis / row_count,
        "min": col_min[i] if obs > 0 else None,
        "max": col_max[i] if obs > 0 else None,
        "mean": mean_val,
        "is_constant": is_const
    })

profiles_path = os.path.join(RESULTS_DIR, "feature_profiles.json")
with open(profiles_path, "w", encoding="utf-8") as f:
    json.dump(feature_profiles, f, indent=2, ensure_ascii=False)
print(f"-> Đã lưu: {profiles_path}")

# 2. Xuất audit_summary.json
audit_summary = {
    "file_audit": {
        "file_name": "aps_failure_training_set.csv",
        "file_size_bytes": file_size,
        "sha256": train_sha256,
        "encoding": bom_status,
        "preamble_lines_count": len(preamble_lines),
        "header_row_0based_index": header_line_index,
        "header_columns_count": len(header),
        "feature_columns_count": len(feature_names),
        "data_rows_count": row_count,
        "malformed_rows_count": malformed_rows
    },
    "target_audit": {
        "column_name": "class",
        "classes_counts": dict(target_counts),
        "pos_count": target_counts["pos"],
        "neg_count": target_counts["neg"],
        "pos_ratio": target_counts["pos"] / row_count,
        "imbalance_ratio": f"{target_counts['neg'] / target_counts['pos']:.2f}:1",
        "invalid_tokens_count": 0
    },
    "missingness_audit": {
        "missing_token": "na",
        "total_missing_values": total_missing,
        "total_observed_values": total_observed,
        "overall_missing_rate": overall_missing_rate,
        "zero_missing_columns_count": len(zero_missing_cols),
        "zero_missing_columns": zero_missing_cols,
        "all_missing_columns_count": len(all_missing_cols),
        "all_missing_columns": all_missing_cols,
        "gt_50pct_missing_columns_count": len(gt_50_cols),
        "gt_70pct_missing_columns_count": len(gt_70_cols),
        "gt_80pct_missing_columns_count": len(gt_80_cols),
        "rows_all_missing_count": rows_all_missing,
        "min_missing_per_row": min_missing_row,
        "max_missing_per_row": max_missing_row
    },
    "zero_variance_audit": {
        "constant_columns_count": len(constant_cols),
        "constant_columns": [[fn, val, obs] for fn, val, obs, _ in constant_cols]
    },
    "duplicates_audit": {
        "total_unique_feature_patterns": unique_patterns,
        "duplicate_groups_count": duplicate_groups,
        "total_duplicate_rows": total_duplicate_rows,
        "surplus_duplicate_rows": surplus_duplicate_rows,
        "conflicting_label_groups_count": len(conflicting_groups),
        "conflicting_groups": conflicting_groups
    },
    "protocol_gates": {
        "p01_integrity_pass": True,
        "p03_duplicates_found": (duplicate_groups > 0),
        "p03_conflicts_found": (len(conflicting_groups) > 0),
        "p03_split_gate_status": p03_status
    }
}

summary_path = os.path.join(RESULTS_DIR, "audit_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(audit_summary, f, indent=2, ensure_ascii=False)
print(f"-> Đã lưu: {summary_path}")

# 3. Xuất acquisition_manifest.json
manifest = {
    "dataset_id": "scania_aps_uci421",
    "uci_url": "https://archive.ics.uci.edu/dataset/421/aps+failure+at+scania+trucks",
    "final_download_url": "https://archive.ics.uci.edu/static/public/421/aps+failure+at+scania+trucks.zip",
    "doi": "10.24432/C51S51",
    "download_timestamp_utc": "2026-10-25T08:37:03.366093+00:00",
    "license_notes": "CC BY 4.0 (UCI) / Scania CV AB GPLv3-or-later notice in description file",
    "archive_file": {
        "filename": "aps+failure+at+scania+trucks.zip",
        "size_bytes": 56618034,
        "sha256": "5504d0402f54faaf97ac0ca085a621645763f5cfea2eb29c592b057d43d4db89"
    },
    "extracted_files": {
        "aps_failure_description.txt": {
            "size_bytes": 4874,
            "sha256": "57b1eb3c551a678ffc1c8112e5b34c566dd16457642563f58aeb3ca65beeb820"
        },
        "aps_failure_training_set.csv": {
            "size_bytes": file_size,
            "sha256": train_sha256
        }
    },
    "test_set_status": "SEALED inside archive, NOT extracted, NOT parsed, NOT evaluated"
}

manifest_path = os.path.join(RESULTS_DIR, "acquisition_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print(f"-> Đã lưu: {manifest_path}")

# 4. Xuất environment.json
env_data = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "processor": platform.processor(),
    "machine": platform.machine(),
    "executable": sys.executable,
    "working_directory": REPO_ROOT
}
try:
    import pandas
    env_data["pandas_version"] = pandas.__version__
except ImportError:
    env_data["pandas_version"] = None
try:
    import numpy
    env_data["numpy_version"] = numpy.__version__
except ImportError:
    env_data["numpy_version"] = None
try:
    import sklearn
    env_data["sklearn_version"] = sklearn.__version__
except ImportError:
    env_data["sklearn_version"] = None

env_path = os.path.join(RESULTS_DIR, "environment.json")
with open(env_path, "w", encoding="utf-8") as f:
    json.dump(env_data, f, indent=2, ensure_ascii=False)
print(f"-> Đã lưu: {env_path}")

# 5. Xuất audit_execution.txt
exec_log_lines = [
    "=== SCANIA APS SOURCE AUDIT EXECUTION LOG (TASK SC-G2-A) ===",
    f"Execution Date: {datetime.date.today().isoformat()} (Asia/Saigon)",
    "Executor: Antigravity (Data Audit Specialist)",
    "Branch: antigravity/sc-g2-source-audit",
    "Git Commit Base: 6a184cc4093719ff77f65e7303a84d97efe44dbc (PR #13 merge on main)",
    "",
    "[STEP 1: ACQUISITION & PROVENANCE VERIFICATION]",
    "- Source: UCI Machine Learning Repository (Dataset ID: 421)",
    "- Source URL: https://archive.ics.uci.edu/dataset/421/aps+failure+at+scania+trucks",
    "- DOI: 10.24432/C51S51",
    "- Final Download URL: https://archive.ics.uci.edu/static/public/421/aps+failure+at+scania+trucks.zip",
    f"- Train File Size: {file_size} bytes",
    f"- Train SHA-256: {train_sha256}",
    "- Test Set Status: SEALED inside archive, NOT extracted, NOT parsed, NOT evaluated.",
    "",
    "[STEP 2: PREAMBLE & HEADER AUDIT]",
    f"- Preamble lines count: {len(preamble_lines)} lines.",
    f"- Header line index: line {header_line_index + 1} (0-based index {header_line_index}).",
    f"- Total header columns: {len(header)} columns (1 'class' + 170 operational features).",
    f"- Encoding status: {bom_status}.",
    "",
    "[STEP 3: DATA ROWS & TARGET AUDIT]",
    f"- Total data rows: {row_count} rows.",
    f"- Malformed rows: {malformed_rows} rows.",
    f"- Target distribution: neg={target_counts['neg']} (98.33%), pos={target_counts['pos']} (1.67%).",
    f"- Class imbalance ratio: {target_counts['neg']/target_counts['pos']:.2f}:1.",
    "- Target token validation: PASS (only 'pos' and 'neg').",
    "",
    "[STEP 4: MISSINGNESS & FEATURE PROFILE AUDIT]",
    "- Missing token: strictly 'na'.",
    f"- Total missing values: {total_missing} ({overall_missing_rate:.4%}).",
    f"- Total observed values: {total_observed} ({total_observed/total_cells:.4%}).",
    f"- Features with 0% missing: {len(zero_missing_cols)} ({zero_missing_cols}).",
    f"- Features with 100% all-missing: {len(all_missing_cols)}.",
    f"- Features with >50% missing: {len(gt_50_cols)}.",
    f"- Features with >70% missing: {len(gt_70_cols)}.",
    f"- Features with >80% missing: {len(gt_80_cols)}.",
    f"- Constant (zero-variance) features: {len(constant_cols)} ('cd_000' = 1209600.0 for 59,324 rows).",
    f"- Row missingness: min={min_missing_row}, max={max_missing_row}, all-missing rows={rows_all_missing}.",
    "",
    "[STEP 5: EXACT-X DUPLICATE & LABEL CONFLICT AUDIT (P03)]",
    f"- Unique feature patterns: {unique_patterns} / {row_count}.",
    f"- Duplicate groups: {duplicate_groups}.",
    f"- Total duplicate rows: {total_duplicate_rows}.",
    f"- Conflicting label groups: {len(conflicting_groups)}.",
    f"- P03 Split Gate Status: {p03_status}.",
    "",
    "[STEP 6: HISTOGRAM FEATURE GROUP AUDIT]",
    f"- Histogram features: {histogram_features_count} (7 families x 10 bins).",
    f"- Counter features: {counter_features_count}.",
    "",
    "[SUMMARY & READINESS GATES]",
    "- READY FOR SCANIA G2 REVIEW: YES",
    f"- READY FOR SCANIA G3: {'YES' if p03_status == 'PASSED' else 'NO'}"
]
log_path = os.path.join(RESULTS_DIR, "audit_execution.txt")
with open(log_path, "w", encoding="utf-8") as f:
    f.write("\n".join(exec_log_lines) + "\n")
print(f"-> Đã lưu: {log_path}")
print("=> TÁI TẠO TOÀN BỘ 5 TỆP ARTIFACTS: HOÀN TẤT THÀNH CÔNG!")

-> Đã lưu: D:\Năm 3\HK1_2026\Nhập môn tối ưu hóa\baitaplon\worktrees\antigravity\results\scania_aps\g2_audit\feature_profiles.json
-> Đã lưu: D:\Năm 3\HK1_2026\Nhập môn tối ưu hóa\baitaplon\worktrees\antigravity\results\scania_aps\g2_audit\audit_summary.json
-> Đã lưu: D:\Năm 3\HK1_2026\Nhập môn tối ưu hóa\baitaplon\worktrees\antigravity\results\scania_aps\g2_audit\acquisition_manifest.json


-> Đã lưu: D:\Năm 3\HK1_2026\Nhập môn tối ưu hóa\baitaplon\worktrees\antigravity\results\scania_aps\g2_audit\environment.json
-> Đã lưu: D:\Năm 3\HK1_2026\Nhập môn tối ưu hóa\baitaplon\worktrees\antigravity\results\scania_aps\g2_audit\audit_execution.txt
=> TÁI TẠO TOÀN BỘ 5 TỆP ARTIFACTS: HOÀN TẤT THÀNH CÔNG!


## 8. Kết luận & Trạng thái Sẵn sàng Cổng Nghiệm thu

- **Tính toàn vẹn & Schema (P01):** Tệp Train 60.000 mẫu, 171 cột đạt 100% chuẩn hợp lệ; biến mục tiêu chỉ chứa `pos` (1.000 mẫu) và `neg` (59.000 mẫu); token missing duy nhất `na`.
- **Kiểm toán trùng lặp (P03):** 60.000 mẫu đặc trưng $X$ duy nhất, 0 nhóm trùng lặp, 0 nhóm xung đột nhãn $\rightarrow$ Cổng P03 đạt `PASSED` (không kích hoạt `BLOCKED`).
- **READY FOR SCANIA G2 REVIEW:** **YES**
- **READY FOR SCANIA G3:** **YES** (Chờ Chủ dự án nghiệm thu G2 và phê duyệt các điểm kỹ thuật để giao Codex bước vào SC-G3-C).